# Preprocessing: Raw Text to Training Tensors

This notebook is the bridge between:

- raw Shakespeare text
- and tensors a decoder-only Transformer can train on.

## Notebook outputs

By the end, this notebook produces:

- tokenizer (`encode` / `decode`)
- vocabulary mappings (`stoi`, `itos`)
- encoded tensors
- train/validation/test split tensors
- chunked autoregressive sequences
- batch generation pipeline (`get_batch`)

And verifies:

- `x.shape == (batch_size, block_size)`
- `y.shape == (batch_size, block_size)`

## MLOps-friendly additions (still notebook-only)

To stay aligned with your proposal while keeping everything in one `.ipynb`, this notebook also includes:

- a central `CONFIG` dictionary for reproducibility
- saved preprocessing metadata and artifacts
- optional Weights & Biases logging hooks (safe to skip if W&B is not configured)

The result is a model-ready preprocessing workflow that your baseline training notebook can plug into directly.

In [11]:
from pathlib import Path
import random
import json

import numpy as np
import pandas as pd
import torch

# -----------------------------
# Central reproducible settings
# -----------------------------
CONFIG = {
    "seed": 42,
    "block_size": 128,
    "batch_size": 32,
    # If True, build splits from a single raw text source.
    # If False, use existing train/validation/test CSV files.
    "build_splits_from_raw_text": False,
    "split_ratios": (0.90, 0.05, 0.05),  # train, val, test
    # Optional W&B hooks (safe if disabled)
    "use_wandb": True,
    "wandb_project": "genre-story-generator",
    "wandb_run_name": "preprocessing-char-level",
}

SEED = CONFIG["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Seed: {SEED}")
print(f"Block size: {CONFIG['block_size']}, Batch size: {CONFIG['batch_size']}")

Using device: cpu
Seed: 42
Block size: 128, Batch size: 32


## Load raw text and define splits

This section supports two modes:

1. **Existing split files** (`train.csv`, `validation.csv`, `test.csv`)  
2. **Single raw corpus split** (if `CONFIG["build_splits_from_raw_text"] = True`)

Using only training text to build the vocabulary avoids leakage from validation/test splits.

In [12]:
# Resolve paths reliably whether notebook is run from repo root or notebooks/.
cwd = Path.cwd()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(f"Could not locate a 'data' directory from cwd: {cwd}")

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = DATA_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

train_path = DATA_DIR / "train.csv"
val_path = DATA_DIR / "validation.csv"
test_path = DATA_DIR / "test.csv"

print(f"Using data dir: {DATA_DIR.resolve()}")

if CONFIG["build_splits_from_raw_text"]:
    # Build train/val/test splits from one continuous corpus.
    # Priority order for source text: train.csv -> input.txt
    if train_path.exists():
        raw_df = pd.read_csv(train_path)
        full_text = "\n".join(raw_df["text"].astype(str).tolist())
    else:
        input_txt = DATA_DIR / "input.txt"
        assert input_txt.exists(), f"Missing source text file: {input_txt}"
        full_text = input_txt.read_text(encoding="utf-8")

    r_train, r_val, r_test = CONFIG["split_ratios"]
    assert abs((r_train + r_val + r_test) - 1.0) < 1e-8, "split_ratios must sum to 1.0"

    n = len(full_text)
    train_end = int(r_train * n)
    val_end = train_end + int(r_val * n)

    train_text = full_text[:train_end]
    val_text = full_text[train_end:val_end]
    test_text = full_text[val_end:]
else:
    # Use prebuilt split files.
    assert train_path.exists(), f"Missing file: {train_path}"
    assert val_path.exists(), f"Missing file: {val_path}"
    assert test_path.exists(), f"Missing file: {test_path}"

    df_train = pd.read_csv(train_path)
    df_val = pd.read_csv(val_path)
    df_test = pd.read_csv(test_path)

    train_text = "\n".join(df_train["text"].astype(str).tolist())
    val_text = "\n".join(df_val["text"].astype(str).tolist())
    test_text = "\n".join(df_test["text"].astype(str).tolist())

print("Train chars:", len(train_text))
print("Val chars:", len(val_text))
print("Test chars:", len(test_text))

Using data dir: C:\Users\utkar\OneDrive\Desktop\Northwestern University\Studies\NLP\Gen AI\genre-story-generator\data
Train chars: 1003854
Val chars: 55770
Test chars: 55770


## Build tokenizer and vocabulary mappings

We use a **character-level tokenizer** for this baseline.

- `stoi`: string-to-index mapping
- `itos`: index-to-string mapping

The vocabulary is built from **training text only** to simulate proper train/validation/test separation.

In [13]:
# Build a character-level vocabulary from training text only.
chars = sorted(list(set(train_text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(text: str):
    """Convert a string into token ids using training vocab."""
    unknown = [ch for ch in text if ch not in stoi]
    if unknown:
        # TinyShakespeare splits usually share charset; this catches mismatch early.
        raise ValueError(f"Found {len(set(unknown))} unseen chars in text split.")
    return [stoi[ch] for ch in text]

def decode(indices):
    """Convert token ids back to a readable string."""
    return "".join(itos[int(i)] for i in indices)

print("Vocab size:", vocab_size)
print("Sample vocab:", chars[:20])

Vocab size: 65
Sample vocab: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G']


In [14]:
# Persist tokenizer mappings + config for reproducibility.
vocab_payload = {
    "vocab_size": vocab_size,
    "chars": chars,
    "stoi": stoi,
    "itos": {str(k): v for k, v in itos.items()},
}

vocab_path = ARTIFACTS_DIR / "char_vocab.json"
with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump(vocab_payload, f, ensure_ascii=False, indent=2)

preprocess_meta = {
    "seed": CONFIG["seed"],
    "block_size": CONFIG["block_size"],
    "batch_size": CONFIG["batch_size"],
    "build_splits_from_raw_text": CONFIG["build_splits_from_raw_text"],
    "split_ratios": list(CONFIG["split_ratios"]),
    "train_chars": len(train_text),
    "val_chars": len(val_text),
    "test_chars": len(test_text),
    "vocab_size": vocab_size,
}

meta_path = ARTIFACTS_DIR / "preprocess_metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(preprocess_meta, f, ensure_ascii=False, indent=2)

print(f"Saved tokenizer mappings to {vocab_path}")
print(f"Saved preprocessing metadata to {meta_path}")

Saved tokenizer mappings to c:\Users\utkar\OneDrive\Desktop\Northwestern University\Studies\NLP\Gen AI\genre-story-generator\data\artifacts\char_vocab.json
Saved preprocessing metadata to c:\Users\utkar\OneDrive\Desktop\Northwestern University\Studies\NLP\Gen AI\genre-story-generator\data\artifacts\preprocess_metadata.json


In [15]:
# Encode each split into contiguous integer tensors.
train_ids = torch.tensor(encode(train_text), dtype=torch.long)
val_ids = torch.tensor(encode(val_text), dtype=torch.long)
test_ids = torch.tensor(encode(test_text), dtype=torch.long)

print("Encoded tensors:")
print("train_ids:", train_ids.shape, train_ids.dtype)
print("val_ids:", val_ids.shape, val_ids.dtype)
print("test_ids:", test_ids.shape, test_ids.dtype)

torch.save(train_ids, ARTIFACTS_DIR / "train_ids.pt")
torch.save(val_ids, ARTIFACTS_DIR / "val_ids.pt")
torch.save(test_ids, ARTIFACTS_DIR / "test_ids.pt")
print(f"Saved encoded tensors to {ARTIFACTS_DIR}")

Encoded tensors:
train_ids: torch.Size([1003854]) torch.int64
val_ids: torch.Size([55770]) torch.int64
test_ids: torch.Size([55770]) torch.int64
Saved encoded tensors to c:\Users\utkar\OneDrive\Desktop\Northwestern University\Studies\NLP\Gen AI\genre-story-generator\data\artifacts


## Chunking and batch generation

For autoregressive training:

- `x` contains token windows of length `block_size`
- `y` is `x` shifted by one token (next-token targets)

This mirrors the exact objective used by your Transformer training notebook.

In [16]:
# Chunk sampler and batch generator.
block_size = CONFIG["block_size"]
batch_size = CONFIG["batch_size"]

def get_batch(split: str, batch_size: int = batch_size, block_size: int = block_size, device: str = device):
    """Sample random contiguous chunks for next-token prediction."""
    if split == "train":
        data = train_ids
    elif split == "val":
        data = val_ids
    elif split == "test":
        data = test_ids
    else:
        raise ValueError("split must be one of {'train', 'val', 'test'}")

    if len(data) <= block_size:
        raise ValueError(f"{split} split too short for block_size={block_size}")

    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

x, y = get_batch("train")
print("x shape:", tuple(x.shape))
print("y shape:", tuple(y.shape))

x shape: (32, 128)
y shape: (32, 128)


In [17]:
# Optional iterator-style batch pipeline for training loops.
def batch_iterator(split: str, num_batches: int, batch_size: int = batch_size, block_size: int = block_size, device: str = device):
    """Yield (x, y) batches repeatedly; useful for model training loops."""
    for _ in range(num_batches):
        yield get_batch(split, batch_size=batch_size, block_size=block_size, device=device)

# Smoke test one batch from iterator
x_iter, y_iter = next(batch_iterator("train", num_batches=1))
print("Iterator batch x shape:", tuple(x_iter.shape))
print("Iterator batch y shape:", tuple(y_iter.shape))

Iterator batch x shape: (32, 128)
Iterator batch y shape: (32, 128)


In [18]:
# Verify autoregressive alignment: y is x shifted by one token.
sample_row = 0
print("Decoded x sample:")
print(decode(x[sample_row][:80].cpu().tolist()))
print("-" * 60)
print("Decoded y sample (shifted by 1):")
print(decode(y[sample_row][:80].cpu().tolist()))

Decoded x sample:
 passage.

Second Watchman:
Ay, wherefore else guard we his royal tent,
But to d
------------------------------------------------------------
Decoded y sample (shifted by 1):
passage.

Second Watchman:
Ay, wherefore else guard we his royal tent,
But to de


## Optional W&B logging hooks (proposal-aligned)

This cell logs preprocessing metadata and key artifacts to Weights & Biases **if enabled**.

- Set `CONFIG["use_wandb"] = True`
- Ensure `wandb` is installed and authenticated
- Local run files are written to `data/wandb/` (not under `notebooks/`)

If disabled, the cell safely skips logging.

In [19]:
if CONFIG["use_wandb"]:
    try:
        import os
        import wandb

        # Store local W&B run files under data/wandb (repo data level, not notebooks/)
        WANDB_DIR = PROJECT_ROOT / "data" / "wandb"
        WANDB_DIR.mkdir(parents=True, exist_ok=True)
        os.environ["WANDB_DIR"] = str(WANDB_DIR)

        run = wandb.init(
            project=CONFIG["wandb_project"],
            name=CONFIG["wandb_run_name"],
            config=CONFIG,
            job_type="preprocessing",
            dir=str(WANDB_DIR),
        )

        # Log high-level preprocessing stats.
        wandb.log(
            {
                "preprocess/vocab_size": vocab_size,
                "preprocess/train_chars": len(train_text),
                "preprocess/val_chars": len(val_text),
                "preprocess/test_chars": len(test_text),
                "preprocess/block_size": block_size,
                "preprocess/batch_size": batch_size,
            }
        )

        # Log saved artifacts (vocab + metadata + tensors) for reproducibility.
        preprocess_artifact = wandb.Artifact("preprocessing-artifacts", type="dataset")
        preprocess_artifact.add_file(str(vocab_path))
        preprocess_artifact.add_file(str(meta_path))
        preprocess_artifact.add_file(str(ARTIFACTS_DIR / "train_ids.pt"))
        preprocess_artifact.add_file(str(ARTIFACTS_DIR / "val_ids.pt"))
        preprocess_artifact.add_file(str(ARTIFACTS_DIR / "test_ids.pt"))
        run.log_artifact(preprocess_artifact)

        wandb.finish()
        print("W&B preprocessing logs and artifacts uploaded.")
    except Exception as e:
        print(f"W&B logging skipped due to error: {e}")
else:
    print("W&B logging disabled. Set CONFIG['use_wandb'] = True to enable.")

wandb: WARNING Artifact "preprocessing-artifacts" already exists with the same content. No new version will be created.


preprocess/batch_size,▁
preprocess/block_size,▁
preprocess/test_chars,▁
preprocess/train_chars,▁
preprocess/val_chars,▁
preprocess/vocab_size,▁
preprocess/batch_size,32
preprocess/block_size,128
preprocess/test_chars,55770
preprocess/train_chars,1003854
preprocess/val_chars,55770


W&B preprocessing logs and artifacts uploaded.


## Completion Check

If all cells above ran successfully, this notebook is model-ready and proposal-aligned:

- tokenizer (`encode`, `decode`) and vocab mappings (`stoi`, `itos`) built
- train/val/test text processed into encoded tensors and saved
- reusable chunk/batch pipeline available through `get_batch(split)` and `batch_iterator(...)`
- shape contract verified:
  - `x.shape == (batch_size, block_size)`
  - `y.shape == (batch_size, block_size)`
- preprocessing metadata + artifacts saved for reproducibility
- optional W&B logging hooks included for experiment tracking